In [2]:
import os
import sys
import torch
import pandas as pd
from tqdm.auto import tqdm
from torch_geometric.data import HeteroData

# 1. Setup paths
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_manager.data_loader import HeteroDataLoader
from src.data_manager.graph_builder import HeteroGraphBuilder

## Graph Loading Pipeline

#### 1- Remove Problematic Graphs Data

In [9]:
# ==================================================
# 1. Initialize Pipeline
# ==================================================
print("="*60)
print("INITIALIZING HETEROGENEOUS GRAPH PIPELINE")
print("="*60)

loader = HeteroDataLoader("../configs/base.yaml")
builder = HeteroGraphBuilder(loader.config)

print(f"✅ DataLoader initialized")
print(f"✅ GraphBuilder initialized")

# ==================================================
# 2. Load All Samples
# ==================================================
print("\n" + "="*60)
print("LOADING TRAINING SAMPLES")
print("="*60)

train_samples = loader.load_all_samples("train")

if not train_samples:
    print("❌ ERROR: No training samples found!")
    print("Check if data directory exists at: data/train/")
    exit(1)

print(f"✅ Loaded {len(train_samples)} samples")

# ==================================================
# 3. FILTER OUT PROBLEMATIC SAMPLES
# ==================================================
print("\n" + "="*60)
print("FILTERING SAMPLES WITH EDGE-NODE MISMATCHES")
print("="*60)

valid_samples = []
problematic_samples = []

for i, sample in enumerate(tqdm(train_samples, desc="Validating samples", unit="sample")):
    sample_name = sample.get('sample_name', f'sample_{i}')
    
    # Skip if no nodes
    if sample["nodes"]["beam"].empty and sample["nodes"]["column"].empty:
        print(f"[SKIP] {sample_name}: No nodes found")
        problematic_samples.append((sample_name, "No nodes"))
        continue
    
    # Check edge-node consistency
    edges_df = sample["edges_raw"]
    if edges_df.empty:
        print(f"[SKIP] {sample_name}: No edges found")
        problematic_samples.append((sample_name, "Empty edge file"))
        continue
    
    # Collect all node names
    all_node_names = set()
    for node_type in ["beam", "column"]:
        df = sample["nodes"][node_type]
        if not df.empty and "Element_Name" in df.columns:
            all_node_names.update(df["Element_Name"].astype(str).tolist())
    
    if not all_node_names:
        print(f"[SKIP] {sample_name}: No node names found")
        problematic_samples.append((sample_name, "No node names"))
        continue
    
    # Check if edges reference existing nodes
    edge_sources = edges_df["Source"].astype(str).tolist()
    edge_targets = edges_df["Target"].astype(str).tolist()
    all_edge_nodes = set(edge_sources + edge_targets)
    
    # Count matches
    matching_nodes = [node for node in all_edge_nodes if node in all_node_names]
    match_percentage = len(matching_nodes) / len(all_edge_nodes) * 100 if all_edge_nodes else 0
    
    if match_percentage < 90:  # Less than 90% of edge nodes match feature nodes
        print(f"[SKIP] {sample_name}: Only {match_percentage:.1f}% of edge nodes match feature nodes")
        problematic_samples.append((sample_name, f"{match_percentage:.1f}% edge match"))
    else:
        valid_samples.append(sample)
        if match_percentage < 100:
            print(f"[WARNING] {sample_name}: {match_percentage:.1f}% edge match (some mismatches)")

print(f"\n📊 Filtering Results:")
print(f"  Total samples: {len(train_samples)}")
print(f"  Valid samples: {len(valid_samples)}")
print(f"  Problematic samples: {len(problematic_samples)}")

if problematic_samples:
    print(f"\n❌ Problematic samples (first 10):")
    for i, (sample_name, reason) in enumerate(problematic_samples[:10]):
        print(f"  {i+1}. {sample_name}: {reason}")
    if len(problematic_samples) > 10:
        print(f"  ... and {len(problematic_samples) - 10} more")

INITIALIZING HETEROGENEOUS GRAPH PIPELINE
✅ DataLoader initialized
✅ GraphBuilder initialized

LOADING TRAINING SAMPLES
✅ Loaded 254 samples

FILTERING SAMPLES WITH EDGE-NODE MISMATCHES


Validating samples:  31%|███▏      | 80/254 [00:00<00:00, 793.87sample/s]

[SKIP] sample_1: Only 79.8% of edge nodes match feature nodes
[SKIP] sample_10: Only 59.2% of edge nodes match feature nodes
[SKIP] sample_102: Only 84.6% of edge nodes match feature nodes
[SKIP] sample_103: Only 64.3% of edge nodes match feature nodes
[WARNING] sample_104: 97.9% edge match (some mismatches)
[SKIP] sample_105: Only 48.1% of edge nodes match feature nodes
[SKIP] sample_106: Only 0.0% of edge nodes match feature nodes
[SKIP] sample_107: Only 57.5% of edge nodes match feature nodes
[SKIP] sample_108: Only 77.8% of edge nodes match feature nodes
[SKIP] sample_109: Only 80.0% of edge nodes match feature nodes
[SKIP] sample_11: Only 78.7% of edge nodes match feature nodes
[SKIP] sample_110: Only 71.0% of edge nodes match feature nodes
[SKIP] sample_111: Only 40.0% of edge nodes match feature nodes
[WARNING] sample_112: 97.9% edge match (some mismatches)
[WARNING] sample_114: 97.8% edge match (some mismatches)
[SKIP] sample_115: Only 71.4% of edge nodes match feature nodes
[W

Validating samples: 100%|██████████| 254/254 [00:00<00:00, 737.05sample/s]

[SKIP] sample_243: Only 81.9% of edge nodes match feature nodes
[WARNING] sample_246: 97.3% edge match (some mismatches)
[WARNING] sample_247: 93.3% edge match (some mismatches)
[WARNING] sample_248: 95.9% edge match (some mismatches)
[SKIP] sample_249: Only 84.5% of edge nodes match feature nodes
[SKIP] sample_25: Only 86.5% of edge nodes match feature nodes
[WARNING] sample_251: 97.3% edge match (some mismatches)
[SKIP] sample_252: Only 36.4% of edge nodes match feature nodes
[SKIP] sample_253: Only 72.2% of edge nodes match feature nodes
[WARNING] sample_254: 94.5% edge match (some mismatches)
[SKIP] sample_27: Only 62.2% of edge nodes match feature nodes
[SKIP] sample_28: Only 84.1% of edge nodes match feature nodes
[SKIP] sample_29: Only 84.5% of edge nodes match feature nodes
[SKIP] sample_3: Only 73.0% of edge nodes match feature nodes
[SKIP] sample_30: Only 82.1% of edge nodes match feature nodes
[SKIP] sample_31: Only 88.9% of edge nodes match feature nodes
[SKIP] sample_32: O

In [12]:
# ==================================================
# 4. BUILD GRAPHS FOR VALID SAMPLES ONLY
# ==================================================
print("\n" + "="*60)
print("BUILDING HETEROGENEOUS GRAPHS (VALID SAMPLES ONLY)")
print("="*60)

all_hetero_graphs = []
success_count = 0
fail_count = 0
failed_samples_details = []

# Reset scalers for fresh start
builder.scalers = {"beam": None, "column": None}

for i, sample in enumerate(tqdm(valid_samples, desc="Processing Graphs", unit="graph")):
    sample_name = sample.get('sample_name', f'valid_sample_{i}')
    
    try:
        # Build the graph
        g = builder.build_hetero_graph(sample)
        
        # Skip graphs with no edges (shouldn't happen with our filtering)
        if g.num_edges == 0:
            print(f"[WARNING] {sample_name}: Graph has 0 edges despite validation")
            # Still add it if you want, or skip:
            # continue
        
        all_hetero_graphs.append(g)
        success_count += 1
        
        # Print success for first few samples
        if success_count <= 5:
            print(f"[SUCCESS #{success_count}] {sample_name}")
            print(f"  - Nodes: {g.num_nodes} (beam: {g['beam'].num_nodes if 'beam' in g else 0}, "
                  f"column: {g['column'].num_nodes if 'column' in g else 0})")
            print(f"  - Edges: {g.num_edges}")
            print(f"  - Features: {g['beam'].x.shape[1] if 'beam' in g else 'N/A'} dim")
        
    except Exception as e:
        print(f"\n❌ ERROR for {sample_name}: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        fail_count += 1
        failed_samples_details.append((sample_name, str(e)))

# ==================================================
# 5. FINAL STATISTICS
# ==================================================
print("\n" + "="*60)
print("FINAL PROCESSING SUMMARY")
print("="*60)

print(f"\n📊 Sample Statistics:")
print(f"  Total samples loaded: {len(train_samples)}")
print(f"  Valid samples after filtering: {len(valid_samples)}")
print(f"  Problematic samples filtered out: {len(problematic_samples)}")
print(f"  Graphs successfully built: {success_count}")
print(f"  Graphs failed during building: {fail_count}")
print(f"  Final success rate: {success_count/len(valid_samples)*100:.1f}%")

if failed_samples_details:
    print(f"\n❌ Failed during graph building ({fail_count}):")
    for i, (sample_name, error) in enumerate(failed_samples_details[:5]):
        print(f"  {i+1}. {sample_name}: {error[:100]}...")
    if len(failed_samples_details) > 5:
        print(f"  ... and {len(failed_samples_details) - 5} more")

if all_hetero_graphs:
    print(f"\n" + "="*60)
    print("GRAPH DATASET STATISTICS")
    print("="*60)
    
    # Overall statistics
    total_nodes = sum(g.num_nodes for g in all_hetero_graphs)
    total_edges = sum(g.num_edges for g in all_hetero_graphs)
    avg_nodes = total_nodes / len(all_hetero_graphs)
    avg_edges = total_edges / len(all_hetero_graphs)
    
    print(f"\n📊 Overall Dataset Stats:")
    print(f"  Total graphs: {len(all_hetero_graphs)}")
    print(f"  Total nodes: {total_nodes}")
    print(f"  Total edges: {total_edges}")
    print(f"  Average nodes per graph: {avg_nodes:.1f}")
    print(f"  Average edges per graph: {avg_edges:.1f}")
    print(f"  Edge-to-node ratio: {avg_edges/avg_nodes:.2f}")
    
    # Check feature consistency
    feature_dims = set()
    for g in all_hetero_graphs:
        for node_type in g.node_types:
            if hasattr(g[node_type], 'x'):
                feature_dims.add(g[node_type].x.shape[1])
    
    print(f"\n🔧 Feature Dimensions:")
    print(f"  Unique feature dimensions: {feature_dims}")
    if len(feature_dims) == 1:
        print(f"  ✅ All graphs have consistent feature dimension: {next(iter(feature_dims))}")
    else:
        print(f"  ⚠️ Warning: Graphs have different feature dimensions!")
        print(f"     Consider standardizing to a common dimension")
    
    # Sample graph details
    print(f"\n" + "="*60)
    print("FIRST GRAPH DETAILS (for verification)")
    print("="*60)
    
    if all_hetero_graphs:
        test_g = all_hetero_graphs[0]
        sample_name = valid_samples[0].get('sample_name', 'sample_0') if valid_samples else "unknown"
        
        print(f"\nSample: {sample_name}")
        
        # Node Information
        node_stats = []
        for n_type in test_g.node_types:
            if hasattr(test_g[n_type], 'x'):
                node_stats.append({
                    "Node Type": n_type,
                    "Nodes": test_g[n_type].num_nodes,
                    "Features": test_g[n_type].x.shape[1],
                    "Labels": test_g[n_type].y.shape[1] if hasattr(test_g[n_type], 'y') else 0
                })
        
        if node_stats:
            print("\nNODE DATA:")
            print(pd.DataFrame(node_stats).to_string(index=False))
        
        # Edge Information
        if hasattr(test_g, 'edge_types') and test_g.edge_types:
            edge_stats = []
            for e_type in test_g.edge_types:
                if hasattr(test_g[e_type], 'edge_index'):
                    edge_stats.append({
                        "Relation": f"{e_type[0]} -> {e_type[2]}",
                        "Edge Type": e_type[1],
                        "Count": test_g[e_type].num_edges
                    })
            
            if edge_stats:
                print("\nEDGE DATA:")
                print(pd.DataFrame(edge_stats).to_string(index=False))
        
        print(f"\n📐 Graph Properties:")
        print(f"  Has isolated nodes: {test_g.has_isolated_nodes()}")
        print(f"  Has self-loops: {test_g.has_self_loops()}")


BUILDING HETEROGENEOUS GRAPHS (VALID SAMPLES ONLY)


Processing Graphs:   1%|          | 1/131 [00:00<00:22,  5.84graph/s]


[DEBUG] Building graph for sample_100
[DEBUG] beam dataframe shape: (205, 43)
[DEBUG] beam columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'Roof_Area_Difference_Excluding_Roof', 'Relative_to_Horizontal_Axis', 'Relative_to_Vertical_Axis', 'Beam_Arrangement', 'Width_to_Length_Ratio', 'Number_of_Horizontal_Frames', 'Number_of_Vertical_Frames', 'Frame_Ratio_X_to_Y', 'Structure_Height', 'Number_of_Floors', 'Earthquake_Coefficient', 'Allowable_Drift', 'Beam_Concrete_Compressive_Strength', 'Column_Concrete_Compressive_Strength', 'Dynamic_Analysis', 'Static_Analysis', 'X1', 'Y1', 'Z1', 'X2', 'Y2', 'Z2', 'Length', 'Angle', 'Story', 'Ele_Type', 'Boundary', 'Alignment', 'GravityForce', 'EX', 'EY', 'CMX', 'CRX', 'DeltaX', 'CMY', 'CRY', 'DeltaY', 'CRXi', 'CRYi']
[DEBUG] Looking for column 'Element_Name' in beam: True
[DEBUG] column dataframe shape: (109, 43)
[DEBUG] column columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'R

Processing Graphs:   8%|▊         | 10/131 [00:00<00:03, 30.59graph/s]

[SUCCESS #3] sample_104
  - Nodes: 184 (beam: 0, column: 0)
  - Edges: 290
  - Features: N/A dim

[DEBUG] Building graph for sample_112
[DEBUG] beam dataframe shape: (162, 43)
[DEBUG] beam columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'Roof_Area_Difference_Excluding_Roof', 'Relative_to_Horizontal_Axis', 'Relative_to_Vertical_Axis', 'Beam_Arrangement', 'Width_to_Length_Ratio', 'Number_of_Horizontal_Frames', 'Number_of_Vertical_Frames', 'Frame_Ratio_X_to_Y', 'Structure_Height', 'Number_of_Floors', 'Earthquake_Coefficient', 'Allowable_Drift', 'Beam_Concrete_Compressive_Strength', 'Column_Concrete_Compressive_Strength', 'Dynamic_Analysis', 'Static_Analysis', 'X1', 'Y1', 'Z1', 'X2', 'Y2', 'Z2', 'Length', 'Angle', 'Story', 'Ele_Type', 'Boundary', 'Alignment', 'GravityForce', 'EX', 'EY', 'CMX', 'CRX', 'DeltaX', 'CMY', 'CRY', 'DeltaY', 'CRXi', 'CRYi']
[DEBUG] Looking for column 'Element_Name' in beam: True
[DEBUG] column dataframe shape: (76, 43)
[DEBUG]

...


[DEBUG] Building graph for sample_49
[DEBUG] beam dataframe shape: (46, 43)
[DEBUG] beam columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'Roof_Area_Difference_Excluding_Roof', 'Relative_to_Horizontal_Axis', 'Relative_to_Vertical_Axis', 'Beam_Arrangement', 'Width_to_Length_Ratio', 'Number_of_Horizontal_Frames', 'Number_of_Vertical_Frames', 'Frame_Ratio_X_to_Y', 'Structure_Height', 'Number_of_Floors', 'Earthquake_Coefficient', 'Allowable_Drift', 'Beam_Concrete_Compressive_Strength', 'Column_Concrete_Compressive_Strength', 'Dynamic_Analysis', 'Static_Analysis', 'X1', 'Y1', 'Z1', 'X2', 'Y2', 'Z2', 'Length', 'Angle', 'Story', 'Ele_Type', 'Boundary', 'Alignment', 'GravityForce', 'EX', 'EY', 'CMX', 'CRX', 'DeltaX', 'CMY', 'CRY', 'DeltaY', 'CRXi', 'CRYi']
[DEBUG] Looking for column 'Element_Name' in beam: True
[DEBUG] column dataframe shape: (28, 43)
[DEBUG] column columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'Roof

Processing Graphs:  96%|█████████▌| 126/131 [00:05<00:00, 28.67graph/s]


[DEBUG] Building graph for sample_72
[DEBUG] beam dataframe shape: (103, 43)
[DEBUG] beam columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'Roof_Area_Difference_Excluding_Roof', 'Relative_to_Horizontal_Axis', 'Relative_to_Vertical_Axis', 'Beam_Arrangement', 'Width_to_Length_Ratio', 'Number_of_Horizontal_Frames', 'Number_of_Vertical_Frames', 'Frame_Ratio_X_to_Y', 'Structure_Height', 'Number_of_Floors', 'Earthquake_Coefficient', 'Allowable_Drift', 'Beam_Concrete_Compressive_Strength', 'Column_Concrete_Compressive_Strength', 'Dynamic_Analysis', 'Static_Analysis', 'X1', 'Y1', 'Z1', 'X2', 'Y2', 'Z2', 'Length', 'Angle', 'Story', 'Ele_Type', 'Boundary', 'Alignment', 'GravityForce', 'EX', 'EY', 'CMX', 'CRX', 'DeltaX', 'CMY', 'CRY', 'DeltaY', 'CRXi', 'CRYi']
[DEBUG] Looking for column 'Element_Name' in beam: True
[DEBUG] column dataframe shape: (32, 43)
[DEBUG] column columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'Roo

Processing Graphs: 100%|██████████| 131/131 [00:06<00:00, 21.81graph/s]


[DEBUG] Building graph for sample_98
[DEBUG] beam dataframe shape: (311, 43)
[DEBUG] beam columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'Roof_Area_Difference_Excluding_Roof', 'Relative_to_Horizontal_Axis', 'Relative_to_Vertical_Axis', 'Beam_Arrangement', 'Width_to_Length_Ratio', 'Number_of_Horizontal_Frames', 'Number_of_Vertical_Frames', 'Frame_Ratio_X_to_Y', 'Structure_Height', 'Number_of_Floors', 'Earthquake_Coefficient', 'Allowable_Drift', 'Beam_Concrete_Compressive_Strength', 'Column_Concrete_Compressive_Strength', 'Dynamic_Analysis', 'Static_Analysis', 'X1', 'Y1', 'Z1', 'X2', 'Y2', 'Z2', 'Length', 'Angle', 'Story', 'Ele_Type', 'Boundary', 'Alignment', 'GravityForce', 'EX', 'EY', 'CMX', 'CRX', 'DeltaX', 'CMY', 'CRY', 'DeltaY', 'CRXi', 'CRYi']
[DEBUG] Looking for column 'Element_Name' in beam: True
[DEBUG] column dataframe shape: (125, 43)
[DEBUG] column columns: ['Element_Name', 'Structural_System', 'Opening_Status', 'Height_Difference', 'Ro

#### Dimensional mismatch throughout the graphs

In [5]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter

print("="*60)
print("ANALYZING FEATURE DIMENSIONS ACROSS ALL GRAPHS")
print("="*60)

# Track feature dimensions
feature_dims = []
dimension_counts = defaultdict(list)  # dim -> [sample_names]
column_names_per_sample = {}  # sample_name -> {beam_columns, column_columns}

# Check each graph
for i, graph in enumerate(all_hetero_graphs):
    sample_name = f"graph_{i}"  # Adjust if you have actual sample names
    dim_info = {}
    
    for node_type in graph.node_types:
        if hasattr(graph[node_type], 'x'):
            dim = graph[node_type].x.shape[1]
            dim_info[node_type] = dim
            
            # Store sample name by dimension
            dimension_counts[dim].append((sample_name, node_type))
    
    feature_dims.append(dim_info)

# Analyze results
print(f"\nTotal graphs analyzed: {len(feature_dims)}")

# Count dimensions
dim_counter = Counter()
for dim_info in feature_dims:
    for node_type, dim in dim_info.items():
        dim_counter[dim] += 1

print("\nFeature Dimension Distribution:")
print("-" * 40)
for dim, count in sorted(dim_counter.items()):
    print(f"  Dimension {dim}: {count} node-type occurrences")

# Find which samples have which dimensions
print("\nSamples by Feature Dimension:")
print("-" * 40)
for dim in sorted(dimension_counts.keys()):
    samples = dimension_counts[dim]
    print(f"\nDimension {dim} ({len(samples)} occurrences):")
    
    # Group by sample
    sample_groups = defaultdict(list)
    for sample, node_type in samples:
        sample_groups[sample].append(node_type)
    
    # Show first 5 samples for each dimension
    for sample, node_types in list(sample_groups.items())[:5]:
        print(f"  - {sample}: {node_types}")
    
    if len(sample_groups) > 5:
        print(f"  ... and {len(sample_groups) - 5} more samples")

# Check if dimensions match within each graph
print("\n" + "="*60)
print("CHECKING DIMENSION CONSISTENCY WITHIN GRAPHS")
print("="*60)

inconsistent_graphs = []
for i, dim_info in enumerate(feature_dims):
    if len(set(dim_info.values())) > 1:
        inconsistent_graphs.append((i, dim_info))

if inconsistent_graphs:
    print(f"⚠️ Found {len(inconsistent_graphs)} graphs with inconsistent dimensions:")
    for graph_idx, dim_info in inconsistent_graphs[:5]:  # Show first 5
        print(f"  Graph {graph_idx}: {dim_info}")
else:
    print("✅ All graphs have consistent dimensions across node types")

ANALYZING FEATURE DIMENSIONS ACROSS ALL GRAPHS

Total graphs analyzed: 254

Feature Dimension Distribution:
----------------------------------------
  Dimension 41: 6 node-type occurrences
  Dimension 49: 502 node-type occurrences

Samples by Feature Dimension:
----------------------------------------

Dimension 41 (6 occurrences):
  - graph_8: ['beam', 'column']
  - graph_158: ['beam', 'column']
  - graph_240: ['beam', 'column']

Dimension 49 (502 occurrences):
  - graph_0: ['beam', 'column']
  - graph_1: ['beam', 'column']
  - graph_2: ['beam', 'column']
  - graph_3: ['beam', 'column']
  - graph_4: ['beam', 'column']
  ... and 246 more samples

CHECKING DIMENSION CONSISTENCY WITHIN GRAPHS
✅ All graphs have consistent dimensions across node types


## Graphs WITHOUT positional encoding (No edges)

In [7]:
print("="*60)
print("IDENTIFYING GRAPHS MISSING POSITIONAL ENCODING")
print("="*60)

# Find graphs with 41 features (no PE)
graphs_without_pe = []
graphs_with_pe = []

for i, graph in enumerate(all_hetero_graphs):
    # Check feature dimension for any node type
    for node_type in graph.node_types:
        if hasattr(graph[node_type], 'x'):
            dim = graph[node_type].x.shape[1]
            if dim == 41:
                graphs_without_pe.append(i)
                break
            elif dim == 49:
                graphs_with_pe.append(i)
                break

print(f"\nGraphs WITH positional encoding (49 features): {len(graphs_with_pe)}")
print(f"Graphs WITHOUT positional encoding (41 features): {len(graphs_without_pe)}")

if graphs_without_pe:
    print(f"\nGraphs missing PE (indices): {graphs_without_pe}")
    
    # Try to get sample names if possible
    print("\nDetails of graphs without PE:")
    for idx in graphs_without_pe[:10]:  # Show first 10
        if idx < len(train_samples):
            sample = train_samples[idx]
            sample_name = sample.get('sample_name', f'sample_{idx+1}')
            
            # Check graph statistics
            graph = all_hetero_graphs[idx]
            print(f"\n  Graph {idx} ({sample_name}):")
            print(f"    - Total nodes: {graph.num_nodes}")
            print(f"    - Total edges: {graph.num_edges}")
            
            # Check if it's a small or disconnected graph
            if graph.num_nodes < 10:
                print(f"    ⚠️ Very small graph (less than 10 nodes)")
            if graph.num_edges == 0:
                print(f"    ⚠️ Graph has no edges")
                
            # Check node type distribution
            for node_type in graph.node_types:
                if hasattr(graph[node_type], 'x'):
                    print(f"    - {node_type}: {graph[node_type].num_nodes} nodes, {graph[node_type].x.shape[1]} features")

IDENTIFYING GRAPHS MISSING POSITIONAL ENCODING

Graphs WITH positional encoding (49 features): 251
Graphs WITHOUT positional encoding (41 features): 3

Graphs missing PE (indices): [8, 158, 240]

Details of graphs without PE:

  Graph 8 (sample_106):
    - Total nodes: 290
    - Total edges: 0
    ⚠️ Graph has no edges
    - beam: 179 nodes, 41 features
    - column: 111 nodes, 41 features

  Graph 158 (sample_241):
    - Total nodes: 350
    - Total edges: 0
    ⚠️ Graph has no edges
    - beam: 239 nodes, 41 features
    - column: 111 nodes, 41 features

  Graph 240 (sample_87):
    - Total nodes: 264
    - Total edges: 0
    ⚠️ Graph has no edges
    - beam: 180 nodes, 41 features
    - column: 84 nodes, 41 features


In [8]:
print("="*60)
print("DEBUGGING EDGE PROCESSING FOR PROBLEMATIC SAMPLES")
print("="*60)

problematic_samples = ["sample_106", "sample_241", "sample_87"]

for sample_name in problematic_samples:
    print(f"\n{'='*40}")
    print(f"Analyzing: {sample_name}")
    print('='*40)
    
    # Load the sample
    sample_data = loader.load_sample(sample_name, "train")
    if not sample_data:
        print(f"❌ Could not load sample {sample_name}")
        continue
    
    # 1. Check edge file content
    edges_df = sample_data["edges_raw"]
    print(f"Edge file: {len(edges_df)} rows")
    print(f"Edge columns: {list(edges_df.columns)}")
    print(f"First few edges:")
    print(edges_df.head())
    
    # 2. Check node data
    print(f"\nNode counts:")
    print(f"  Beams: {len(sample_data['nodes']['beam'])}")
    print(f"  Columns: {len(sample_data['nodes']['column'])}")
    
    # 3. Check if Element_Name values match
    beam_names = sample_data["nodes"]["beam"]["Element_Name"].tolist()
    column_names = sample_data["nodes"]["column"]["Element_Name"].tolist()
    all_node_names = set(beam_names + column_names)
    
    print(f"\nNode name ranges:")
    print(f"  Beam names: {min(beam_names) if beam_names else 'N/A'} to {max(beam_names) if beam_names else 'N/A'}")
    print(f"  Column names: {min(column_names) if column_names else 'N/A'} to {max(column_names) if column_names else 'N/A'}")
    
    # 4. Check edge source/target values
    edge_sources = edges_df["Source"].unique()
    edge_targets = edges_df["Target"].unique()
    all_edge_nodes = set(edge_sources.tolist() + edge_targets.tolist())
    
    print(f"\nEdge node ranges:")
    print(f"  Source range: {min(edge_sources) if len(edge_sources) > 0 else 'N/A'} to {max(edge_sources) if len(edge_sources) > 0 else 'N/A'}")
    print(f"  Target range: {min(edge_targets) if len(edge_targets) > 0 else 'N/A'} to {max(edge_targets) if len(edge_targets) > 0 else 'N/A'}")
    
    # 5. Check for matching issues
    matching_sources = [src for src in edge_sources if src in all_node_names]
    matching_targets = [tgt for tgt in edge_targets if tgt in all_node_names]
    
    print(f"\nEdge-to-node matching:")
    print(f"  Total unique edge nodes: {len(all_edge_nodes)}")
    print(f"  Edge nodes found in node list: {len([n for n in all_edge_nodes if n in all_node_names])}")
    print(f"  Sources matching nodes: {len(matching_sources)}/{len(edge_sources)}")
    print(f"  Targets matching nodes: {len(matching_targets)}/{len(edge_targets)}")
    
    # 6. Find specific mismatches
    if len(matching_sources) < len(edge_sources):
        mismatched = [src for src in edge_sources if src not in all_node_names]
        print(f"\n  ❌ Sources NOT in node list (first 5): {mismatched[:5]}")
    
    if len(matching_targets) < len(edge_targets):
        mismatched = [tgt for tgt in edge_targets if tgt not in all_node_names]
        print(f"  ❌ Targets NOT in node list (first 5): {mismatched[:5]}")

DEBUGGING EDGE PROCESSING FOR PROBLEMATIC SAMPLES

Analyzing: sample_106
Edge file: 180 rows
Edge columns: ['Source', 'Target']
First few edges:
   Source  Target
0       6      40
1      40       6
2       6      41
3      41       6
4       6      51

Node counts:
  Beams: 179
  Columns: 111

Node name ranges:
  Beam names: 105 to 371
  Column names: 82 to 367

Edge node ranges:
  Source range: 6 to 72
  Target range: 6 to 72

Edge-to-node matching:
  Total unique edge nodes: 30
  Edge nodes found in node list: 0
  Sources matching nodes: 0/30
  Targets matching nodes: 0/30

  ❌ Sources NOT in node list (first 5): [6, 40, 41, 51, 52]
  ❌ Targets NOT in node list (first 5): [40, 6, 41, 51, 52]

Analyzing: sample_241
Edge file: 30 rows
Edge columns: ['Source', 'Target']
First few edges:
   Source  Target
0     334     336
1     336     334
2     334     337
3     337     334
4     334     342

Node counts:
  Beams: 239
  Columns: 111

Node name ranges:
  Beam names: 27 to 697
  Column 